# Resume YOLOv8 training: epoch 6 -> 10 on Colab

Use a free T4 to continue the existing run for **4 more epochs** (epochs 7-10) without overheating the laptop.

## 1. Upload to Google Drive

Upload these two files from the project root:

- `training/colab_resume_epoch6.zip`
- `dataset/pigfeces-yolo.zip`

## 2. Run

1. **Runtime -> Change runtime type -> GPU (T4)**.
2. Run all cells in order.
3. At the end, download `training-resume-outputs.zip` from Drive.
4. Install locally with:

```bash
python training/install_colab_weights.py <path-to>/training-resume-outputs.zip --force
```


In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# 2) Configuration
from pathlib import Path

DRIVE_BUNDLE = '/content/drive/MyDrive/colab_resume_epoch6.zip'
DRIVE_DATASET = '/content/drive/MyDrive/pigfeces-yolo.zip'
DRIVE_OUT = '/content/drive/MyDrive/training-resume-outputs.zip'

TARGET_TOTAL_EPOCHS = 10
CHUNK_EPOCHS = 4

REPO = Path('/content/feces-detector-ai')
RUN_DIR = REPO / 'training' / 'runs' / 'detect'
DATA_YAML = REPO / 'training' / 'dataset.yaml'

assert Path(DRIVE_BUNDLE).exists(), f'Missing {DRIVE_BUNDLE}'
assert Path(DRIVE_DATASET).exists(), f'Missing {DRIVE_DATASET}'
print('bundle:', Path(DRIVE_BUNDLE).stat().st_size, 'bytes')
print('dataset:', Path(DRIVE_DATASET).stat().st_size, 'bytes')

In [ ]:
# 3) Unzip dataset, locate the dataset root, write dataset.yaml, verify classes
import glob, shutil, zipfile, yaml

DATASET_DIR = REPO / 'dataset'
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(DRIVE_DATASET) as z:
    z.extractall(DATASET_DIR)
    zip_names = z.namelist()

candidates = [
    p.parent.parent
    for p in DATASET_DIR.rglob('train')
    if p.is_dir()
    and p.parent.name == 'images'
    and (p.parent.parent / 'labels' / 'train').is_dir()
]
candidates = sorted(set(candidates), key=lambda p: str(p))
if not candidates:
    tops = sorted({n.split('/')[0] for n in zip_names if n})
    raise SystemExit(
        'ABORT: no images/train + labels/train found in the dataset zip. '
        f'Top-level entries: {tops[:10]}. '
        'Upload the converted 2-class dataset/pigfeces-yolo.zip, not the resume bundle or raw Roboflow export.'
    )
if len(candidates) > 1:
    print('multiple dataset roots found; using the first:', candidates)
DATASET_ROOT = candidates[0]
print('dataset root:', DATASET_ROOT)

for d in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    p = DATASET_ROOT / d
    print(d, len(list(p.iterdir())) if p.is_dir() else 'MISSING')

yaml_text = f'''path: {DATASET_ROOT.as_posix()}
train: images/train
val: images/val
names:
  0: feces
  1: pig
'''
DATA_YAML.parent.mkdir(parents=True, exist_ok=True)
DATA_YAML.write_text(yaml_text)
print(yaml_text)

cfg = yaml.safe_load(DATA_YAML.read_text())
expected = {int(k) for k in cfg['names']}
per_class = {i: 0 for i in expected}
for split in ('train', 'val'):
    for lbl in glob.glob(str(DATASET_ROOT / 'labels' / split / '*.txt')):
        with open(lbl) as f:
            for line in f:
                parts = line.split()
                if not parts:
                    continue
                cls = int(float(parts[0]))
                if cls not in expected:
                    raise SystemExit(f'ABORT: {lbl} has class id {cls}, expected {sorted(expected)}')
                per_class[cls] += 1
print('boxes per class:', per_class)
assert all(per_class.values()), f'Missing class labels: {per_class}'

In [ ]:
# 4) Install Ultralytics and check the GPU
!pip -q install 'ultralytics==8.4.123'
import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise SystemExit('GPU not enabled: Runtime -> Change runtime type -> GPU (T4)')

In [ ]:
# 5) Unpack the existing run (last.pt, best.pt, results.csv)
import zipfile
RUN_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DRIVE_BUNDLE) as z:
    print(z.namelist())
    z.extractall(RUN_DIR)
print('run files:', sorted(p.name for p in RUN_DIR.iterdir()))

In [ ]:
# 6) Resume for 4 more epochs, then stop automatically
import torch
from ultralytics import YOLO

CKPT = RUN_DIR / 'weights' / 'last.pt'
assert CKPT.exists(), CKPT

ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
start_epoch = ckpt['epoch'] + 1
checkpoint_total = ckpt['train_args']['epochs']
stop_at = min(TARGET_TOTAL_EPOCHS, checkpoint_total)
chunk_end = min(start_epoch + CHUNK_EPOCHS, stop_at)
print(f'checkpoint completed {start_epoch} epochs; chunk stops at epoch {chunk_end}')

model = YOLO(str(CKPT))

def on_train_start(trainer):
    print(f'Resuming at epoch {trainer.start_epoch + 1}/{trainer.epochs}; chunk stops after epoch {chunk_end}')

def on_fit_epoch_end(trainer):
    print(f'Epoch {trainer.epoch + 1}/{trainer.epochs} complete')
    if trainer.epoch + 1 >= chunk_end:
        trainer.stop = True
        print(f'Chunk complete at epoch {trainer.epoch + 1}')

model.add_callback('on_train_start', on_train_start)
model.add_callback('on_fit_epoch_end', on_fit_epoch_end)

model.train(
    resume=True,
    data=str(DATA_YAML),
    save_dir=str(RUN_DIR),
    device=0,
    batch=6,
    imgsz=640,
    workers=2,
    verbose=True,
)

In [ ]:
# 7) Verify the run reached 10/60
import csv, torch

rows = list(csv.DictReader(open(RUN_DIR / 'results.csv', encoding='utf-8')))
print('results rows:', len(rows))
for r in rows[-4:]:
    print(r['epoch'], r.get('metrics/mAP50(B)'))

ckpt = torch.load(RUN_DIR / 'weights' / 'last.pt', map_location='cpu', weights_only=False)
completed = ckpt['epoch'] + 1
print('checkpoint completed epochs:', completed, 'of', ckpt['train_args']['epochs'])
assert completed >= TARGET_TOTAL_EPOCHS, f'Only {completed} epochs completed'
assert len(rows) >= TARGET_TOTAL_EPOCHS, f'results.csv only has {len(rows)} rows'
print('TARGET REACHED:', completed, 'epochs')

In [ ]:
# 8) Package outputs and copy back to Drive
import shutil, zipfile

out = '/content/training-resume-outputs.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('best.pt', 'last.pt'):
        p = RUN_DIR / 'weights' / name
        if p.exists():
            z.write(p, f'weights/{name}')
    for name in ('results.csv', 'evaluation_summary.txt'):
        p = RUN_DIR / name
        if p.exists():
            z.write(p, name)
shutil.copy(out, DRIVE_OUT)
print('DOWNLOAD READY:', DRIVE_OUT)

## Install locally

Download `training-resume-outputs.zip` from Drive, then run from the project root:

```bash
python training/install_colab_weights.py <path-to>/training-resume-outputs.zip --force
```

The local run is then at **10/60**. To continue later, make a new bundle from the updated `last.pt` and repeat with `TARGET_TOTAL_EPOCHS = 14`.
